# Real Belgian PV forecasts in a few cells

Use the published Python package to fetch real Elia PV estimates, run three simple
baselines and compare forecasts interactively. No repository checkout, local research
bundle, weather preparation, model training or weights are required.

Use **Python 3.11–3.13**. **Run all cells in order.** The download cell makes a bounded, explicit network download
for **1–14 June 2026, end exclusive**. The three evaluation issues are fixed before
looking at errors: 10–12 June at 11:00 Europe/Brussels, forecasting the next local day.
The earlier observations provide warm-up history; this is not a training split.


## 1. Install into this notebook's kernel

Skip this cell if these versions are already installed. Restart the kernel if an
older SolarCast or pandas version was imported before installation. Plotly and
nbformat are notebook display dependencies, separate from SolarCast's base package.


In [ ]:
%pip install -q "solarcast-belgium==0.1.0" "plotly>=6,<7" "nbformat>=5,<6"

In [ ]:
import pandas as pd
import plotly.express as px
from solarcast import BenchmarkConfig, benchmark
from solarcast.data import EliaPVClient

## 2. Fetch real national PV estimates

[Elia historical source](https://opendata.elia.be/explore/dataset/ods032/) provides
quarter-hour national measurements/upscaled estimates in MW. SolarCast returns
interval starts in UTC, keeps missing observations missing and retains source and
retrieval timestamps. This demonstration does not redistribute or save the data.
See the [source contract and licensing caveat](https://github.com/Junaid-Bashir/solarcast-belgium/blob/v0.1.0/docs/data/elia.md).


In [ ]:
with EliaPVClient() as client:
    pv = client.fetch_history(start="2026-06-01T00:00:00+02:00", end="2026-06-14T00:00:00+02:00")

pv.head()

## 3. Run the shared backtest

The 48-hour and seven-day baselines reuse exact elapsed-UTC seasonal lags, without
fallback. Last-value persistence holds the most recent eligible observation.
All three receive the same bounded history and are scored on common next-day targets,
including night. The leaderboard reports missing forecasts/failures as well as errors.

**Information-availability assumption:** the explicit 30-minute safety lag is only
an approximation. Fetching today's retrospective Elia snapshot cannot reconstruct
historical revisions or prove what was published at each issue. This short example
is a usage demonstration, not confirmatory or operational performance evidence.


In [ ]:
config = BenchmarkConfig(
    issue_times=pd.date_range("2026-06-10 11:00", periods=3, freq="D", tz="Europe/Brussels"),
    observation_safety_lag="30min",
    reference_model="seasonal_persistence_48h",
)
results = benchmark(
    data=pv,
    config=config,
    models=["seasonal_persistence_48h", "seasonal_persistence_7d", "last_value_persistence"],
    data_source="Elia historical national PV; live retrospective snapshot",
)
results.leaderboard()

## 4. Explore forecasts against actual generation

Hover to inspect MW values; zoom or drag the range slider; click legend entries to
show/hide a series. Times below are Europe/Brussels; the stored data remain UTC.
The chart covers the same evaluation targets as the leaderboard. It does not drop
failed forecasts, fill missing values or clip negative values. Scoring actuals are
joined here for display only; they are not supplied as future model inputs.


In [ ]:
profiles = results.predictions.pivot(
    index="target_time", columns="model_id", values="prediction_mw"
)
profiles["Elia actual"] = pv.set_index("target_time")["pv_generation_mw"]
profiles.index = profiles.index.tz_convert("Europe/Brussels")

fig = px.line(
    profiles,
    labels={"target_time": "Target time (Europe/Brussels)", "value": "PV generation (MW)"},
    title="Real Belgian PV: next-day baseline forecasts and Elia actuals",
)
fig.update_layout(hovermode="x unified", xaxis_rangeslider_visible=True)
fig.show()

## What next?

Change the explicit dates and keep enough preceding history for every selected lag.
Brussels target days can contain 92, 96 or 100 quarter-hours around DST; never assume
a fixed count. Changing model choices after seeing these errors is exploratory work,
not an untouched evaluation.

For trainable models, see the [public API guide](https://github.com/Junaid-Bashir/solarcast-belgium/blob/main/docs/public-api.md):
LightGBM needs an explicit chronological fit schedule; TSFMs need optional runtimes
and separately acquired weights; weather-informed models need a prepared local
historical WeatherStore. None is downloaded implicitly by this notebook.

The separate `01_real_belgian_forecast_analysis.ipynb` explores the already-frozen
five-model research comparison and requires its original local research bundles.
